In [19]:
# !pip install torch torchvision torchaudio 

In [20]:
import torch
import os
import torch.nn as nn
import pandas as pd
import re
from tqdm import tqdm
tqdm.pandas()
import numpy as np
from glob import glob
import sys
import json
os.environ["NUMBA_LOG_LEVEL"] = "WARNING"


In [21]:
device = torch.device('cuda')

In [22]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)

        div = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div)
        pe[:, 1::2] = torch.cos(position * div)

        pe = pe.unsqueeze(0)  
        self.register_buffer("pe", pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return x


def create_look_ahead_mask(size):
    """
    mask future tokens → True = block
    Shape: (size, size)
    """
    return torch.triu(torch.ones(size, size, dtype=torch.bool), diagonal=1)




class G2PTransformer(nn.Module):
    def __init__(self, input_dim, output_dim,
                 d_model=256, nhead=8,
                 num_encoder_layers=3, num_decoder_layers=3,
                 dim_feedforward=512, dropout=0.1, pad_idx=0):
        super().__init__()

        self.pad_idx = pad_idx

        # embeddings
        self.embedding_src = nn.Embedding(input_dim, d_model, padding_idx=pad_idx)
        self.embedding_tgt = nn.Embedding(output_dim, d_model, padding_idx=pad_idx)
        self.pos_encoder = PositionalEncoding(d_model)

        # encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_encoder_layers)

        # decoder
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_decoder_layers)

        # output linear layer
        self.fc_out = nn.Linear(d_model, output_dim)

    def forward(self, src, tgt):
        """
        src: (B, S)
        tgt: (B, T)
        """

        # padding masks (B, S) & (B, T)
        src_key_padding_mask = (src == self.pad_idx)
        tgt_key_padding_mask = (tgt == self.pad_idx)

        # look ahead mask (T, T)
        seq_len = tgt.size(1)
        tgt_mask = create_look_ahead_mask(seq_len).to(tgt.device)

        # embeddings + positional encoding
        src_emb = self.pos_encoder(self.embedding_src(src))
        tgt_emb = self.pos_encoder(self.embedding_tgt(tgt))

        # encoder
        memory = self.encoder(
            src_emb,
            src_key_padding_mask=src_key_padding_mask
        )

        # decoder
        out = self.decoder(
            tgt_emb,
            memory,
            tgt_mask=tgt_mask,
            tgt_key_padding_mask=tgt_key_padding_mask,
            memory_key_padding_mask=src_key_padding_mask
        )

        return self.fc_out(out)

In [23]:
g2p_model = torch.load("/kaggle/input/models/morvaridrhbr/g2p-full-model/pytorch/default/1/full_G2P_model.pt", weights_only=False)
g2p_model.eval()

G2PTransformer(
  (embedding_src): Embedding(88, 256, padding_idx=0)
  (embedding_tgt): Embedding(33, 256, padding_idx=0)
  (pos_encoder): PositionalEncoding()
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-3): 4 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
        )
        (linear1): Linear(in_features=256, out_features=512, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=512, out_features=256, bias=True)
        (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (decoder): TransformerDecoder(
    (layers): ModuleList(
      (0-3): 4 x TransformerDecoderLayer(
        (self_attn): MultiheadAtte

<h4>Transcribtion process</h4>

In [24]:
metadata_pth = '/kaggle/input/datasets/magnoliasis/persian-tts-dataset-famale/metadata.csv'
text = pd.read_csv(metadata_pth, sep="|", header=None, names=["text", "filename"])
text

,text,filename
0,کوتولههای سفید میشوند . ستارگان سنگینتر بهصورت...,0.wav
1,جهتیابی با درختان از انجا که سمتِ شمالیِ درختا...,1.wav
2,ترکتِ کرده ؟ شب ها ب ه یاد ش گریه م ی کنی ؟ ن ...,2.wav
3,برایِ دیدن این عکس با اندازهِ واقعیِ اینجا کلی...,3.wav
4,من از طرفدارانِ م عضِ رِ خواهیِ میکنمِ چون کار...,4.wav
...,...,...
11130,راهِ میافتد . در اینِ حرکتِ اتومبیلها دو بار ا...,11130.wav
11131,نکتهِ تصویرِ مولفهِ هر بردار روی یک محورِ براب...,11131.wav
11132,نکتهِ بیست و شش در ربعِ سوم نوسانگر در جهتِ من...,11132.wav
11133,ضمنا دوستان با توجه به اینکه در مدیریتِ محتوای...,11133.wav


In [25]:
unwanted_chars = [
    '\n', '\u200b','\u200c','\u200e','\u200f','\ufeff','\xa0',
    '!', '"', '#', '&', "'", '(', ')', '+', ',', '-', '.', '/', 
    ':',';','<','=','>','?','@','_','–','“','”','…','«','»','٪','ـ',
    '0','1','2','3','4','5','6','7','8','9','k','n','u'
    '۰','۱','۲','۳','۴','۵','۶','۷','۸','۹', '٠','١','٢','٣','٤','٥','٦','٧','٨','٩',
    '،','؛','؟','٫','٬','ü','ā','š','ʔ','新','最','\xad']

In [26]:
pattern = "[" + "".join(unwanted_chars) + "]"
text['text'] = text['text'].apply(lambda x: re.sub(pattern, '', x))
text

,text,filename
0,کوتولههای سفید میشوند ستارگان سنگینتر بهصورتِ...,0.wav
1,جهتیابی با درختان از انجا که سمتِ شمالیِ درختا...,1.wav
2,ترکتِ کرده شب ها ب ه یاد ش گریه م ی کنی ن ار...,2.wav
3,برایِ دیدن این عکس با اندازهِ واقعیِ اینجا کلی...,3.wav
4,من از طرفدارانِ م عضِ رِ خواهیِ میکنمِ چون کار...,4.wav
...,...,...
11130,راهِ میافتد در اینِ حرکتِ اتومبیلها دو بار از...,11130.wav
11131,نکتهِ تصویرِ مولفهِ هر بردار روی یک محورِ براب...,11131.wav
11132,نکتهِ بیست و شش در ربعِ سوم نوسانگر در جهتِ من...,11132.wav
11133,ضمنا دوستان با توجه به اینکه در مدیریتِ محتوای...,11133.wav


In [27]:
text['text_tokens'] = text['text'].apply(list)
text

,text,filename,text_tokens
0,کوتولههای سفید میشوند ستارگان سنگینتر بهصورتِ...,0.wav,"[ک, و, ت, و, ل, ه, ه, ا, ی, , س, ف, ی, د, , ..."
1,جهتیابی با درختان از انجا که سمتِ شمالیِ درختا...,1.wav,"[ج, ه, ت, ی, ا, ب, ی, , ب, ا, , د, ر, خ, ت, ..."
2,ترکتِ کرده شب ها ب ه یاد ش گریه م ی کنی ن ار...,2.wav,"[ت, ر, ک, ت, ِ, , ک, ر, د, ه, , , ش, ب, , ..."
3,برایِ دیدن این عکس با اندازهِ واقعیِ اینجا کلی...,3.wav,"[ب, ر, ا, ی, ِ, , د, ی, د, ن, , ا, ی, ن, , ..."
4,من از طرفدارانِ م عضِ رِ خواهیِ میکنمِ چون کار...,4.wav,"[م, ن, , ا, ز, , ط, ر, ف, د, ا, ر, ا, ن, ِ, ..."
...,...,...,...
11130,راهِ میافتد در اینِ حرکتِ اتومبیلها دو بار از...,11130.wav,"[ر, ا, ه, ِ, , م, ی, ا, ف, ت, د, , , د, ر, ..."
11131,نکتهِ تصویرِ مولفهِ هر بردار روی یک محورِ براب...,11131.wav,"[ن, ک, ت, ه, ِ, , ت, ص, و, ی, ر, ِ, , م, و, ..."
11132,نکتهِ بیست و شش در ربعِ سوم نوسانگر در جهتِ من...,11132.wav,"[ن, ک, ت, ه, ِ, , ب, ی, س, ت, , و, , ش, ش, ..."
11133,ضمنا دوستان با توجه به اینکه در مدیریتِ محتوای...,11133.wav,"[ض, م, ن, ا, , د, و, س, ت, ا, ن, , ب, ا, , ..."


In [28]:
grapheme_dict = {'<pad>': 0, ' ': 1, 'ء': 2, 'آ': 3, 'أ': 4, 'ؤ': 5, 'إ': 6, 'ئ': 7,
              'ا': 8, 'ب': 9, 'ة': 10, 'ت': 11, 'ث': 12, 'ج': 13, 'ح': 14, 'خ': 15,
              'د': 16, 'ذ': 17, 'ر': 18, 'ز': 19, 'س': 20, 'ش': 21, 'ص': 22, 'ض': 23,
              'ط': 24, 'ظ': 25, 'ع': 26, 'غ': 27, 'ف': 28, 'ق': 29, 'ك': 30, 'ل': 31,
              'م': 32, 'ن': 33, 'ه': 34, 'و': 35, 'ى': 36, 'ي': 37, 'ً': 38, 'ٌ': 39,
              'َ': 40, 'ُ': 41, 'ِ': 42, 'ّ': 43, 'ْ': 44, 'ٔ': 45, 'پ': 46, 'چ': 47,
              'ژ': 48, 'ک': 49, 'گ': 50, 'ۀ': 51, 'ی': 52, 'ے': 53, 'ە': 54, 'ﭘ': 55,
              'ﮐ': 56, 'ﮔ': 57, 'ﯽ': 58, 'ﯾ': 59, 'ﯿ': 60, 'ﺍ': 61, 'ﺎ': 62, 'ﺑ': 63,
              'ﺒ': 64, 'ﺖ': 65, 'ﺘ': 66, 'ﺧ': 67, 'ﺩ': 68, 'ﺪ': 69, 'ﺭ': 70, 'ﺮ': 71,
              'ﺱ': 72, 'ﺴ': 73, 'ﺷ': 74, 'ﺸ': 75, 'ﻀ': 76, 'ﻋ': 77, 'ﻌ': 78, 'ﻟ': 79,
              'ﻡ': 80, 'ﻢ': 81, 'ﻤ': 82, 'ﻥ': 83, 'ﻧ': 84, 'ﻪ': 85, 'ﻭ': 86, 'ﻮ': 87}


In [29]:
def grapheme_toID(tokens , graph_dict):
    return[graph_dict[t] for t in tokens]

In [30]:
text['text_ids'] = text['text_tokens'].apply(lambda x : grapheme_toID(x , grapheme_dict))
text

,text,filename,text_tokens,text_ids
0,کوتولههای سفید میشوند ستارگان سنگینتر بهصورتِ...,0.wav,"[ک, و, ت, و, ل, ه, ه, ا, ی, , س, ف, ی, د, , ...","[49, 35, 11, 35, 31, 34, 34, 8, 52, 1, 20, 28,..."
1,جهتیابی با درختان از انجا که سمتِ شمالیِ درختا...,1.wav,"[ج, ه, ت, ی, ا, ب, ی, , ب, ا, , د, ر, خ, ت, ...","[13, 34, 11, 52, 8, 9, 52, 1, 9, 8, 1, 16, 18,..."
2,ترکتِ کرده شب ها ب ه یاد ش گریه م ی کنی ن ار...,2.wav,"[ت, ر, ک, ت, ِ, , ک, ر, د, ه, , , ش, ب, , ...","[11, 18, 49, 11, 42, 1, 49, 18, 16, 34, 1, 1, ..."
3,برایِ دیدن این عکس با اندازهِ واقعیِ اینجا کلی...,3.wav,"[ب, ر, ا, ی, ِ, , د, ی, د, ن, , ا, ی, ن, , ...","[9, 18, 8, 52, 42, 1, 16, 52, 16, 33, 1, 8, 52..."
4,من از طرفدارانِ م عضِ رِ خواهیِ میکنمِ چون کار...,4.wav,"[م, ن, , ا, ز, , ط, ر, ف, د, ا, ر, ا, ن, ِ, ...","[32, 33, 1, 8, 19, 1, 24, 18, 28, 16, 8, 18, 8..."
...,...,...,...,...
11130,راهِ میافتد در اینِ حرکتِ اتومبیلها دو بار از...,11130.wav,"[ر, ا, ه, ِ, , م, ی, ا, ف, ت, د, , , د, ر, ...","[18, 8, 34, 42, 1, 32, 52, 8, 28, 11, 16, 1, 1..."
11131,نکتهِ تصویرِ مولفهِ هر بردار روی یک محورِ براب...,11131.wav,"[ن, ک, ت, ه, ِ, , ت, ص, و, ی, ر, ِ, , م, و, ...","[33, 49, 11, 34, 42, 1, 11, 22, 35, 52, 18, 42..."
11132,نکتهِ بیست و شش در ربعِ سوم نوسانگر در جهتِ من...,11132.wav,"[ن, ک, ت, ه, ِ, , ب, ی, س, ت, , و, , ش, ش, ...","[33, 49, 11, 34, 42, 1, 9, 52, 20, 11, 1, 35, ..."
11133,ضمنا دوستان با توجه به اینکه در مدیریتِ محتوای...,11133.wav,"[ض, م, ن, ا, , د, و, س, ت, ا, ن, , ب, ا, , ...","[23, 32, 33, 8, 1, 16, 35, 20, 11, 8, 33, 1, 9..."


In [31]:
phoneme_dict = {
    '<pad>': 0,
    ' ': 1,
    '<eos>': 2,
    '<sos>': 3,
    '?': 4,
    'A': 5,
    'C': 6,
    'S': 7,
    'Z': 8,
    'a': 9,
    'b': 10,
    'd': 11,
    'e': 12,
    'f': 13,
    'g': 14,
    'h': 15,
    'i': 16,
    'j': 17,
    'k': 18,
    'l': 19,
    'm': 20,
    'n': 21,
    'o': 22,
    'p': 23,
    'q': 24,
    'r': 25,
    's': 26,
    't': 27,
    'u': 28,
    'v': 29,
    'x': 30,
    'y': 31,
    'z': 32
}


In [32]:
text = text.dropna(subset=['text']).reset_index(drop=True)
# text

In [33]:
all_inputs = text['text_ids'].tolist()

In [34]:
def g2p_batch_inference(
    input_texts,
    model,
    phoneme_dict,
    device,
    max_len=100,
    batch_size=32
):
    model.eval()

    sos_id = phoneme_dict['<sos>']
    eos_id = phoneme_dict['<eos>']
    pad_id = phoneme_dict['<pad>']

    results = []

    for i in tqdm(range(0, len(input_texts), batch_size)):
        batch = input_texts[i:i+batch_size]

        
        src = [torch.tensor(x, dtype=torch.long) for x in batch]
        src = torch.nn.utils.rnn.pad_sequence(src, batch_first=True, padding_value=pad_id)
        src = src.to(device)

        
        tgt = torch.full((src.size(0), 1), sos_id, dtype=torch.long).to(device)

        finished = [False] * src.size(0)
        outputs = [[] for _ in range(src.size(0))]

        with torch.no_grad():
            for _ in range(max_len):
                out = model(src, tgt)
                next_token = out[:, -1, :].argmax(-1).unsqueeze(1)
                tgt = torch.cat([tgt, next_token], dim=1)

                for b in range(src.size(0)):
                    if not finished[b]:
                        token = next_token[b].item()
                        if token == eos_id:
                            finished[b] = True
                        else:
                            outputs[b].append(token)

                if all(finished):
                    break

        results.extend(outputs)

    return results


text['phoneme_tokens'] = g2p_batch_inference(
    text['text'].tolist(),
    g2p_model,
    phoneme_dict,
    device,
    batch_size=128
)


In [35]:
text['phoneme_tokens'] = g2p_batch_inference(
    text['text_ids'].tolist(),
    g2p_model,
    phoneme_dict,
    device,
    batch_size=256
)

100%|██████████| 44/44 [09:51<00:00, 13.45s/it]


In [36]:
text

,text,filename,text_tokens,text_ids,phoneme_tokens
0,کوتولههای سفید میشوند ستارگان سنگینتر بهصورتِ...,0.wav,"[ک, و, ت, و, ل, ه, ه, ا, ی, , س, ف, ی, د, , ...","[49, 35, 11, 35, 31, 34, 34, 8, 52, 1, 20, 28,...","[18, 28, 27, 28, 19, 12, 15, 5, 31, 12, 1, 26,..."
1,جهتیابی با درختان از انجا که سمتِ شمالیِ درختا...,1.wav,"[ج, ه, ت, ی, ا, ب, ی, , ب, ا, , د, ر, خ, ت, ...","[13, 34, 11, 52, 8, 9, 52, 1, 9, 8, 1, 16, 18,...","[17, 9, 15, 9, 27, 31, 5, 10, 16, 1, 10, 5, 1,..."
2,ترکتِ کرده شب ها ب ه یاد ش گریه م ی کنی ن ار...,2.wav,"[ت, ر, ک, ت, ِ, , ک, ر, د, ه, , , ش, ب, , ...","[11, 18, 49, 11, 42, 1, 49, 18, 16, 34, 1, 1, ...","[27, 22, 25, 18, 9, 27, 12, 1, 18, 9, 25, 11, ..."
3,برایِ دیدن این عکس با اندازهِ واقعیِ اینجا کلی...,3.wav,"[ب, ر, ا, ی, ِ, , د, ی, د, ن, , ا, ی, ن, , ...","[9, 18, 8, 52, 42, 1, 16, 52, 16, 33, 1, 8, 52...","[10, 9, 25, 5, 31, 12, 1, 11, 16, 11, 9, 21, 1..."
4,من از طرفدارانِ م عضِ رِ خواهیِ میکنمِ چون کار...,4.wav,"[م, ن, , ا, ز, , ط, ر, ف, د, ا, ر, ا, ن, ِ, ...","[32, 33, 1, 8, 19, 1, 24, 18, 28, 16, 8, 18, 8...","[20, 9, 21, 1, 4, 9, 32, 1, 27, 9, 25, 9, 13, ..."
...,...,...,...,...,...
11130,راهِ میافتد در اینِ حرکتِ اتومبیلها دو بار از...,11130.wav,"[ر, ا, ه, ِ, , م, ی, ا, ف, ت, د, , , د, ر, ...","[18, 8, 34, 42, 1, 32, 52, 8, 28, 11, 16, 1, 1...","[25, 5, 15, 12, 1, 20, 16, 22, 13, 27, 9, 11, ..."
11131,نکتهِ تصویرِ مولفهِ هر بردار روی یک محورِ براب...,11131.wav,"[ن, ک, ت, ه, ِ, , ت, ص, و, ی, ر, ِ, , م, و, ...","[33, 49, 11, 34, 42, 1, 11, 22, 35, 52, 18, 42...","[21, 22, 18, 27, 12, 31, 12, 1, 27, 9, 26, 29,..."
11132,نکتهِ بیست و شش در ربعِ سوم نوسانگر در جهتِ من...,11132.wav,"[ن, ک, ت, ه, ِ, , ب, ی, س, ت, , و, , ش, ش, ...","[33, 49, 11, 34, 42, 1, 9, 52, 20, 11, 1, 35, ...","[21, 22, 18, 27, 12, 31, 12, 1, 10, 16, 26, 27..."
11133,ضمنا دوستان با توجه به اینکه در مدیریتِ محتوای...,11133.wav,"[ض, م, ن, ا, , د, و, س, ت, ا, ن, , ب, ا, , ...","[23, 32, 33, 8, 1, 16, 35, 20, 11, 8, 33, 1, 9...","[32, 12, 20, 21, 5, 1, 11, 22, 26, 27, 5, 21, ..."


In [37]:
id2phoneme = {v: k for k, v in phoneme_dict.items()}

In [38]:
def phoneme_ids_to_symbols(token_ids, id2phoneme):
    return [id2phoneme.get(i, '<unk>') for i in token_ids]

In [39]:
text['phoneme_tokens'] = text['phoneme_tokens'].apply(
    lambda x: phoneme_ids_to_symbols(x, id2phoneme)
)
text.to_csv('/kaggle/working/meta.csv',encoding='utf-8')

<h4>Data Preperation</h4>


In [40]:
text = text.sample(frac=1, random_state=42).reset_index(drop=True)

val_size = int(len(text) * 0.1) 

val_df = text.iloc[:val_size]
train_df = text.iloc[val_size:]

print(len(train_df), len(val_df))


10022 1113


In [41]:
def phoneme_list_to_string(ph_list):
    if isinstance(ph_list, str):
        ph_list = ph_list.strip("[]").replace(",", "").split()

    return " ".join([p for p in ph_list if p != ""])


In [42]:
text

,text,filename,text_tokens,text_ids,phoneme_tokens
0,کوتاهترینِ جنگ در تاریخ در سالِ یکِ هزار و هشت...,6114.wav,"[ک, و, ت, ا, ه, ت, ر, ی, ن, ِ, , ج, ن, گ, , ...","[49, 35, 11, 8, 34, 11, 18, 52, 33, 42, 1, 13,...","[k, u, t, A, h, t, a, r, i, n, , j, a, n, g, ..."
1,اینجا ست که بعضی وقتها ادم ارزو میکنه کاش یهِ ...,483.wav,"[ا, ی, ن, ج, ا, , س, ت, , ک, ه, , ب, ع, ض, ...","[8, 52, 33, 13, 8, 1, 20, 11, 1, 49, 34, 1, 9,...","[?, i, n, j, A, , s, e, t, , k, e, , b, a, ..."
2,ستارهشناسان تا کنون بیش از یکصد و شصت سیاره بی...,8410.wav,"[س, ت, ا, ر, ه, ش, ن, ا, س, ا, ن, , ت, ا, , ...","[20, 11, 8, 18, 34, 21, 33, 8, 20, 8, 33, 1, 1...","[s, e, t, A, r, e, S, e, n, A, s, A, n, , t, ..."
3,هوشیار و بیدارِ سازِ وهم در اینِ روزِ جرم و گن...,8042.wav,"[ه, و, ش, ی, ا, ر, , و, , ب, ی, د, ا, ر, ِ, ...","[34, 35, 21, 52, 8, 18, 1, 35, 1, 9, 52, 16, 8...","[h, u, S, i, y, A, r, , v, a, , b, i, d, A, ..."
4,تو اتوبوس نشستم با رانندهِ گرمِ گرفتم ازم پرسی...,6849.wav,"[ت, و, , ا, ت, و, ب, و, س, , ن, ش, س, ت, م, ...","[11, 35, 1, 8, 11, 35, 9, 35, 20, 1, 33, 21, 2...","[t, o, , ?, o, t, o, b, u, s, , n, e, S, a, ..."
...,...,...,...,...,...
11130,عصرِ کهِ رفتیم پیش دکتر به ش گفتمِ اون م گفت ...,5734.wav,"[ع, ص, ر, ِ, , ک, ه, ِ, , ر, ف, ت, ی, م, , ...","[26, 22, 18, 42, 1, 49, 34, 42, 1, 18, 28, 11,...","[?, a, s, r, e, , k, e, , r, o, f, t, i, m, ..."
11131,هر شرکتی جعبه دنده و شتاب دهنده را بصورتِ هایِ...,5191.wav,"[ه, ر, , ش, ر, ک, ت, ی, , ج, ع, ب, ه, , د, ...","[34, 18, 1, 21, 18, 49, 11, 52, 1, 13, 26, 9, ...","[h, a, r, , S, e, r, k, a, t, i, , j, a, ?, ..."
11132,انگاه از خوراکِ انان میل فرمود و اظهار داشت من...,5390.wav,"[ا, ن, گ, ا, ه, , ا, ز, , خ, و, ر, ا, ک, ِ, ...","[8, 33, 50, 8, 34, 1, 8, 19, 1, 15, 35, 18, 8,...","[?, a, n, g, A, h, , ?, a, z, , x, o, r, A, ..."
11133,جواهر چیان که متولدِ سالِ یکِ هزار و سیصد و پن...,860.wav,"[ج, و, ا, ه, ر, , چ, ی, ا, ن, , ک, ه, , م, ...","[13, 35, 8, 34, 18, 1, 47, 52, 8, 33, 1, 49, 3...","[j, a, v, A, h, e, r, , C, i, y, A, n, , k, ..."


In [43]:
wav_dir = "/kaggle/input/datasets/magnoliasis/persian-tts-dataset-famale/wavs"  

def write_manifest(df, output_path):
    with open(output_path, "w", encoding="utf-8") as f:
        for _, row in df.iterrows():

            wav_path = os.path.join(wav_dir, row["filename"])

            if isinstance(row["phoneme_tokens"], list):
                phonemes = " ".join(row["phoneme_tokens"])
            else:
                phonemes = row["phoneme_tokens"]

            f.write(f"{wav_path}|{phonemes}\n")


write_manifest(train_df, "/kaggle/working/train.txt")
write_manifest(val_df, "/kaggle/working/val.txt")